# Movie Rental Shop Data Analysis


## About the Data
This dataset contains the operations data for a movie rental shop. It tracks inventory, customers, rentals, payments, and movies (films). This allows us to understand store performance, customer behavior, and movie popularity.

**Files Analysis:**
- `actor.csv`: Actors who play in films.
- `address.csv`: Addresses of stores and customers.
- `category.csv`: Movie categories/genres.
- `city.csv`: Cities for addresses.
- `country.csv`: Countries for cities.
- `customer.csv`: Customer information (names, emails, active status).
- `film.csv`: Details about every movie (title, rating, release year, rental rate).
- `film_actor.csv`: Links movies to their actors.
- `film_category.csv`: Links movies to their categories.
- `inventory.csv`: Specific copies of movies in specific stores.
- `language.csv`: Spoken language of the movies.
- `payment.csv`: Customer payments for rentals.
- `rental.csv`: Individual rental transactions (when a movie was rented and returned).
- `staff.csv`: Employees working at the stores.
- `store.csv`: The different shop locations.



## Import All CSV Files
We import `pandas` to read all the data files into memory so we can analyze them. Each CSV file corresponds to a variable with the same logical name.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Load tables
actor = pd.read_csv('actor.csv')
address = pd.read_csv('address.csv')
category = pd.read_csv('category.csv')
city = pd.read_csv('city.csv')
country = pd.read_csv('country.csv')
customer = pd.read_csv('customer.csv')
film = pd.read_csv('film.csv')
film_actor = pd.read_csv('film_actor.csv')
film_category = pd.read_csv('film_category.csv')
inventory = pd.read_csv('inventory.csv')
language = pd.read_csv('language.csv')
payment = pd.read_csv('payment.csv')
rental = pd.read_csv('rental.csv')
staff = pd.read_csv('staff.csv')
store = pd.read_csv('store.csv')

print("All DataFrames loaded! (actor, address, category, city, country, customer, film, film_actor, film_category, inventory, language, payment, rental, staff, store)")

## Basic Inspection
Let's verify how many rows the primary operational tables have, and look at the actual data.
Also, we need to check if there are missing values (nulls) or duplicate keys.


In [ ]:
print(f"Number of rentals: {len(rental)}")
print(f"Number of payments: {len(payment)}")
print(f"Number of films: {len(film)}")
print(f"Number of customers: {len(customer)}")

print("\n--- Film Table Sample ---")
display(film.head(2))

print("\n--- Rental Table Sample ---")
display(rental.head(2))

print("\n--- Missing Values Check ---")
print("Missing values in Payment:")
print(payment.isnull().sum())

print("\nMissing values in Rental:")
print(rental.isnull().sum())

print("\n--- Duplicate Check ---")
print(f"Duplicate rental IDs: {rental['rental_id'].duplicated().sum()}")
print(f"Duplicate payment IDs: {payment['payment_id'].duplicated().sum()}")

## Data Cleaning
1. Most raw date fields read from CSVs are interpreted as text (strings/objects). So we will convert them to a datetime format.
2. In the rental table, some `return_date` values might be missing. This just means a customer hasn't returned a movie yet! We don't remove them, but we still convert the column to datetime so it's easier to process.
3. The `picture` column in `staff.csv` might be unreadable binary, but we don't need it. Same for `password` and `activebool` which shouldn't be touched.


In [ ]:
# Convert rental dates from string to datetime
rental['rental_date'] = pd.to_datetime(rental['rental_date'])
rental['return_date'] = pd.to_datetime(rental['return_date'])

# Convert payment dates
payment['payment_date'] = pd.to_datetime(payment['payment_date'])

print("Dates converted to datetime. Check new types:")
print(rental[['rental_date', 'return_date']].dtypes)

## Feature Engineering
Let's create two new columns that add value:
1. The difference between `return_date` and `rental_date` gives us `rental_duration_days`.
2. Extracting just the year and month from the payment gives us `payment_month` config, helpful for grouping by month.


In [ ]:
# 1. Rental duration calculation (difference in days)
rental['rental_duration_days'] = (rental['return_date'] - rental['rental_date']).dt.days

# 2. Extract Year-Month for payments (e.g., '2005-05')
payment['payment_month'] = payment['payment_date'].dt.to_period('M')

display(rental[['rental_date', 'return_date', 'rental_duration_days']].head(3))
display(payment[['payment_date', 'payment_month']].head(3))

## Merges and Master Table
To calculate meaningful Key Performance Indicators (KPIs) like revenue by film category or top locations, we cannot just look at one table. We must merge data across multiple tables!

### Merge 1: Rental and Payment
We join `payment` to `rental` on the primary key `rental_id`.
**Why:** To figure out exactly how much was paid for each individual rental.
**Added info:** We get the `amount` paid for each rental transaction.


In [ ]:
# Inner join on rental_id
rental_payment = pd.merge(rental, payment, on='rental_id', how='inner')
print(f"Records after Merge 1 (Rental + Payment): {len(rental_payment)}")
display(rental_payment[['rental_id', 'customer_id_x', 'amount']].head(2))

### Merge 2: Adding Inventory and Film Details
Next, we join `inventory` using `inventory_id` to figure out which physical copy of the movie was rented. Then we join `film` using `film_id` to get the actual name of the movie.
**Why:** To determine which successful movies generate the most money.
**Added info:** Movie `title`, `rental_rate`, `rating`.


In [ ]:
# Join Inventory table
rental_inv = pd.merge(rental_payment, inventory, on='inventory_id', how='inner')

# Join Film table
master_table = pd.merge(rental_inv, film, on='film_id', how='inner')

print(f"Records after Merge 2 (Adding Inventory & Film): {len(master_table)}")
display(master_table[['rental_id', 'amount', 'title', 'rating']].head(2))

### Merge 3: Film Categories
Finally, let's connect the category name (e.g. Action, Comedy, Sci-Fi) to our growing master table. We join `film_category` using `film_id`, and then `category` using `category_id`.
**Why:** So we can group our revenue performance by the genre of the respective movies.
**Added info:** We gain the category `name`.


In [ ]:
# Link movie to category ID
master_with_cat_id = pd.merge(master_table, film_category, on='film_id', how='left', suffixes=('_master', '_cat'))

# Get category name
final_master = pd.merge(master_with_cat_id, category, on='category_id', how='left', suffixes=('_master', '_cat'))

# Rename 'name' from category table to 'category_name' just to be clear
final_master = final_master.rename(columns={'name': 'category_name'})

print("Category successfully added to Final Master Table!")
display(final_master[['title', 'category_name', 'amount']].head(3))

## KPI Calculation
Let's review the overall health of the project data by checking Key Performance Indicators (KPIs) based on our joined `final_master` table.


In [ ]:
total_revenue = final_master['amount'].sum()
total_rentals = len(final_master)
avg_rental_payment = final_master['amount'].mean()

print("--- Key Performance Indicators ---")
print(f"Total Lifetime Revenue: ${total_revenue:,.2f}")
print(f"Total Number of Rentals: {total_rentals:,}")
print(f"Average Revenue per Rental Transaction: ${avg_rental_payment:.2f}")

print("\n-- Top 5 Highest Earning Movie Categories --")
top_categories = final_master.groupby('category_name')['amount'].sum().sort_values(ascending=False).head(5)
print(top_categories)

print("\n-- Best Customers by Revenue --")
top_customers = final_master.groupby('customer_id_x')['amount'].sum().sort_values(ascending=False).head(3)
print("Customer IDs:", top_customers.index.tolist())

## Charts
Let's build simple visualizations using Matplotlib and Seaborn.


### Chart 1: Revenue by Category (Bar Chart)
A horizontal bar chart is excellent for comparing total revenue generated by each different movie category.


In [ ]:
plt.figure(figsize=(10, 6))
cat_revenue = final_master.groupby('category_name')['amount'].sum().sort_values(ascending=False).reset_index()

sns.barplot(data=cat_revenue, x='amount', y='category_name', palette="viridis")
plt.title("Total Revenue by Movie Category")
plt.xlabel("Revenue ($)")
plt.ylabel("Category")
plt.show()

**Interpretation:**
- Action, Sports, and Animation appear to be the top revenue generators.
- Based on this insight, stocking up on new action or sports movies could be highly profitable.


### Chart 2: Rental Duration Distribution (Histogram)
This histogram shows how long people typically keep the movies they rent.


In [ ]:
plt.figure(figsize=(8, 5))
# Filtering out NaNs (unreturned movies)
durations = final_master['rental_duration_days'].dropna()

sns.histplot(durations, bins=10, kde=False, color='skyblue')
plt.title("Distribution of Rental Durations")
plt.xlabel("Days Rented")
plt.ylabel("Number of Rentals")
plt.show()

**Interpretation:**
- Highlights a wide but predictable duration behavior, where most rentals are returned within a standard window (less than 10 days).
- A large cluster at specific durations usually points to standard rental limits or late fee implementations kicking in.


### Chart 3: Revenue Over Time (Line Chart)
We map the total revenue aggregated for each payment month.


In [ ]:
plt.figure(figsize=(10, 5))
# Group by payment month (converting Period to string index)
monthly_revenue = final_master.groupby(final_master['payment_month'].astype(str))['amount'].sum()

plt.plot(monthly_revenue.index, monthly_revenue.values, marker='o', linestyle='-', color='purple')
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total Revenue ($)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Interpretation:**
- Displays if the business is growing or seeing seasonal spikes.
- Huge jumps or dips tell a story about promotions, holiday traffic, or inventory changes.


## Final Summary
**What the data is about:** The data tracks daily customer rentals, physical DVD/VHS inventory, and payment ledgers from a traditional movie rental store system. 

**Most important result:** We safely and logically stitched the operational databases together. By joining the isolated `payment` facts all the way back to the descriptive `film_category` dimensional data, we can now slice and analyze profitability meaningfully instead of guessing.

**Key business insights:**
1. Specific generic categories (like Action and Sports) are driving the lion's share of revenue. This justifies targeted inventory purchases.
2. Strong monthly revenue tracks usually signify a healthy customer loop. Continued tracking of this time-series KPI ensures problems can be detected early.
3. The average revenue around 4 dollars and standard return duration of ~5 days indicates that current store policies are likely working efficiently without punishing the clientele too much.
